# Hindlimb Model Edits 
+ Use this notebook to keep track of code used to edit the hindlimb model
+ Place functions in the `src` directory and import them in the blocks that need them 
+ Document your changes with markdown cells before the code block in the style of:

## Step 0: Load the model
This step loads the model and sets up the environment for editing.

In [ ]:
import opensim as osim
from src.muscle_utils import attachments_to_csv

# Load the model
model = osim.Model('rat_hindlimb.osim')
attachments_to_csv(model, 'muscle_attachments.csv')

## Step 1: Convert Thelen muscles to Millard
There is an error in the equilibrate muscles function for Thelen muscles, and Millard muscles are more prevalent.

In [ ]:
from src.muscle_utils import model_thelen_to_millard

millard_model = model_thelen_to_millard(model)
# Save the modified model
millard_model.printToXML('rat_hindlimb_millard.osim')

## Step 2: Analyze muscle lengths and moment arms
Create plots of the muscle lengths and moment arms to find discontinuities or other potential errors.

In [ ]:
from src.musculoskeletal_graph import MusculoskeletalGraph

millard_tsl_model = millard_model.clone()
graph = MusculoskeletalGraph(millard_tsl_model)

# Iterate through the range of motion to compute lengths and moment arms for all muscles
results = graph.muscle_rom_analysis(min_points=1000)

In [ ]:
import plotly.graph_objects as go
import os
# import plotly.io as pio
# pio.renderers.default = "notebook"
figure_dir = 'figs'

# Plot lengths and moment arms to identify problems
for muscle_name, muscle_data in results.items():
    # Plot parallel coordinates of muscle lengths
    fig1 = go.Figure(data=
        go.Parcoords(
            line=dict(color=muscle_data['length'], colorscale='Viridis', showscale=True),
            dimensions=[dict(
                label=col,
                values=muscle_data[col],
            )
            for col in muscle_data.columns if not 'moment_arm' in col]
        )   
    )   
    fig1.update_layout(title=muscle_name + ' Lengths')
    # fig1.write_html(f"{figure_dir}/{muscle_name}_length.html")
    if not os.path.exists(f"{figure_dir}/{muscle_name}"):
        os.makedirs(f"{figure_dir}/{muscle_name}")
    fig1.write_image(f"{figure_dir}/{muscle_name}/{muscle_name}_length.png")

    moment_arm_cols = [col for col in muscle_data.columns if 'moment_arm' in col]
    coord_cols = [col for col in muscle_data.columns if not 'moment_arm' in col and col != 'length']
    for moment_arm in moment_arm_cols:
        # Plot scatter plot of moment arm versus singular coordinate
        for coord in coord_cols:
            fig2 = go.Figure(data=
                go.Scatter(
                    x=muscle_data[coord],
                    y=muscle_data[moment_arm],
                    mode='markers',
                    marker=dict(color=muscle_data[moment_arm], colorscale='Viridis', showscale=True),
                )
            )
            fig2.update_layout(title=f'{muscle_name} {moment_arm} vs {coord}')
            # fig2.write_html(f"{figure_dir}/{muscle_name}_{moment_arm}_vs_{coord}.html")
            fig2.write_image(f"{figure_dir}/{muscle_name}/{muscle_name}_{moment_arm}_vs_{coord}.png")

## Step 3: Rough estimate tendon slack lengths
This is done initially so that the model can be opened in OpenSim Creator for easier muscle editing. It will need to be redone later to account for changes to moment arms. This method is currently using the whole range of motion of each joint, but you could also specify coordinate combinations that you expect to see in the analyzed motion. Using the below method, you can catch most issues, but if you want to just focus on a specific motion, it might be unnecessary.

In [ ]:
from src.muscle_utils import optimize_model_tsl

results = graph.muscle_rom_analysis(min_points=10)
millard_tsl_model: osim.Model = optimize_model_tsl(millard_tsl_model, results)

millard_tsl_model.printToXML('rat_hindlimb_millard_tsl.osim')

## Potentially problematic muscles
### BFa 
![BFa Length](./figs/BFa/BFa_length.png)

### CF
![CF Length](./figs/CF/CF_length.png)

### STp
![STp Length](./figs/STp/STp_length.png)

### TA
![TA Length](./figs/TA/TA_length.png)

### VL
![VL Length](./figs/VL/VL_length.png)

Because of the hard-coded value in [OpenSim's WrapCylinder.cpp code](https://github.com/opensim-org/opensim-core/blob/f9cd558ec3ea99dda206e5e412e62e23cf19bd7e/OpenSim/Simulation/Wrap/WrapCylinder.cpp#L668):

```c++
// Each muscle segment on the surface of the cylinder should be
// 0.002 meters long. This assumes the model is in meters, of course.
int numWrapSegments = (int)(aWrapResult.wrap_path_length / 0.002);
if (numWrapSegments < 1) numWrapSegments = 1;
```
we need to modify the wrapping surfaces to prevent discontinuities in muscle lengths that appear in the range of motion of the rats during normal gait

We plan to use the values presented in Young's work. However, the coordinate systems are defined differently and the scale is different. We will need to adjust the values accordingly.

In [8]:
def transform_point(point, scaling_matrix, rigid_matrix, affine_matrix):
    """
    Transform a point using the sequence: scaling → rigid → affine
    
    Args:
        point: NumPy array of shape (3,) representing [x, y, z]
        scaling_matrix, rigid_matrix, affine_matrix: 4x4 transformation matrices
    
    Returns:
        Transformed point as NumPy array of shape (3,)
    """
    # Convert point to homogeneous coordinates
    homogeneous_point = np.append(point, 1)
    
    H = affine_matrix @ rigid_matrix @ scaling_matrix
    print(H)
    # Apply transformations in sequence
    # Step 1: Apply scaling
    scaled_point = scaling_matrix @ homogeneous_point
    
    # Step 2: Apply rigid transformation (rotation + translation)
    rigid_transformed_point = rigid_matrix @ scaled_point
    
    # Step 3: Apply affine transformation
    affine_transformed_point = affine_matrix @ rigid_transformed_point
    
    # Convert back to 3D coordinates
    transformed_point = affine_transformed_point[:3] / affine_transformed_point[3]
    
    return transformed_point

def transform_points(points, scaling_matrix, rigid_matrix, affine_matrix):
    """
    Transform multiple points using the sequence: scaling → rigid → affine
    
    Args:
        points: NumPy array of shape (n, 3) where n is the number of points
        scaling_matrix, rigid_matrix, affine_matrix: 4x4 transformation matrices
    
    Returns:
        Transformed points as NumPy array of shape (n, 3)
    """
    # Add homogeneous coordinate
    homogeneous_points = np.hstack([points, np.ones((points.shape[0], 1))])
    
    # Apply transformations in sequence
    scaled_points = homogeneous_points @ scaling_matrix.T
    rigid_transformed_points = scaled_points @ rigid_matrix.T
    affine_transformed_points = rigid_transformed_points @ affine_matrix.T
    
    # Convert back to 3D coordinates
    transformed_points = affine_transformed_points[:, :3] / affine_transformed_points[:, 3:]
    
    return transformed_points

In [ ]:
import numpy as np
BFa_young = np.array([19.501, -0.571, -5.385])

scaling_matrix = np.array([
    [0.0945188, 0, 0, 0],
    [0, 0.0945188, 0, 0],
    [0, 0, 0.0945188, 0],
    [0, 0, 0, 1.0]
])

rigid_matrix = np.array([
    [-0.101005,  -0.0732987,  0.992182, -0.00549601],
    [0.271701,  -0.961404,  -0.0433655,  -0.00341442],
    [0.957067,  0.265197,  0.117022,  -0.00976972],
    [0,  0,  0,  1]
])

affine_matrix = np.array([
    [0.986489,  -0.00512656,  -0.128231,  -0.00058805],
    [0.00666878,  0.94803,  -0.0229631,  1.60934e-05],
    [-0.0140869,  -0.0144142,  0.825013,  -0.000624756],
    [0,  0,  0,  1]
])

transformed_point = transform_point(BFa_young, scaling_matrix, rigid_matrix, affine_matrix)

print("Transformed point:", transformed_point)



Transformed point: [-0.90235933  0.49846308  1.38727604]


In [ ]:
import numpy as np

BFa_young_tibia = np.array([0.01844, -0.08961, 0.02876])

scaling_matrix = np.array([
    [0.093118, 0, 0, 0],
    [0, 0.093118, 0, 0],
    [0, 0, 0.093118, 0],
    [0, 0, 0, 1]
])

rigid_matrix = np.array([
    [-0.0086619, 0.185862, 0.982538, -0.000924187], 
    [0.00275886, 0.982575, -0.185845, -0.0396265], 
    [-0.999959, 0.00110092, -0.00902373, 4.50859e-05], 
    [0, 0, 0, 1] 
])

inv_rigid_matrix = np.array([
    [-0.0086619, 0.00275886, -0.999959, 0.000146403], 
    [0.185862, 0.982575, 0.00110092, 0.0391077], 
    [0.982538, -0.185845, -0.00902373, -0.00645591], 
    [0, 0, 0, 1]
])

affine_matrix = np.array([
    [1.05521, -0.00590155, -0.0461345, -6.16966e-05], 
    [0.0615527, 1.0001, -0.0337617, -4.41534e-05], 
    [-0.0533887, -0.0054026, 1.1537, -0.000112774], 
    [0, 0, 0, 1] 
])

inv_affine_matrix = np.array([
    [0.949266, 0.00580758, 0.0381295, 6.3123e-05], 
    [-0.0569502, 0.999711, 0.026978, 4.36694e-05], 
    [0.0436616, 0.00495025, 0.868667, 0.000100875], 
    [0, 0, 0, 1] 
])


def transform_point_between_models(point, source_to_target=True):
    """
    Transform a point from one model's coordinate system to another
    
    Args:
        point: NumPy array of shape (3,) representing [x, y, z]
        source_to_target: If True, transform from source model to target model.
                          If False, transform from target model to source model.
    
    Returns:
        Transformed point as NumPy array of shape (3,)
    """
    # Convert to homogeneous coordinates
    homogeneous_point = np.append(point, 1)
    
    if source_to_target:
        # Source model → World → Target model
        # Step 1: Source local to world (apply all transforms)
        point_world = affine_matrix @ rigid_matrix @ scaling_matrix @ homogeneous_point
        
        # Step 2: World to target local (apply inverse of all transforms)
        inv_target_transform = np.linalg.inv(affine_matrix @ rigid_matrix @ scaling_matrix)
        point_target = inv_target_transform @ point_world
    else:
        # Target model → World → Source model
        # Step 1: Target local to world
        target_to_world = affine_matrix @ rigid_matrix @ scaling_matrix
        point_world = target_to_world @ homogeneous_point
        
        # Step 2: World to source local
        world_to_source = np.linalg.inv(affine_matrix @ rigid_matrix @ scaling_matrix)
        point_source = world_to_source @ point_world
        
        point_target = point_source  # Renamed for consistency in return
    
    # Convert back to 3D coordinates
    result = point_target[:3] / point_target[3]
    
    return result


transformed_point = transform_point(BFa_young_tibia, scaling_matrix, inv_rigid_matrix, affine_matrix)
print("Transformed point:", transformed_point)

# From the spreadsheet
spreadsheet_H = np.array([
 [ 0.09521664,  0.14781327, -0.98442116, -0.00038815894],
 [-0.08029481,  0.98683235, 0.14040892, 0.04264428044],
 [ 0.99221295,  0.06567464, 0.1058315,   0.00366359715],
 [ 0.        ,  0.        , 0.          , 1.        ],
 ])

# Apply the transformation
spreadsheet_transformed_point = spreadsheet_H @ np.append(BFa_young_tibia, 1)
print("Transformed spreadsheet point:", spreadsheet_transformed_point)



SyntaxError: invalid syntax (3540698462.py, line 90)

## Final Step: Create Bilateral Model

In [ ]:
# from src import mirror_hindlimb
# #TODO: Implement the mirror_hindlimb function to create a bilateral model
# bilateral_model : osim.Model = mirror_hindlimb(model)

# # Save the mirrored model
# bilateral_model.printToXML('rat_hindlimb_bilateral.osim')